<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/06-adaptation/01-fine-tune-vs-rag-vs-prompt.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 微调、RAG、prompt 三条路线对比

**目标：** 做出 AI 工程师拿钱该做的那次判断：什么时候该*改模型*（fine-tune / LoRA），什么时候该改它的*输入*（prompt、RAG），并且能为这个判断辩护。然后，可选地，在一块免费 GPU 上跑一次真实的 LoRA 微调。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

## 准备工作

概念部分（1–4）和其余每个 notebook 一样，跑在免费的 Groq API 上，下面两个单元格就够了。末尾那个**可选的 LoRA 附录**需要 GPU runtime 和另一套技术栈；它自带依赖安装，并且有清楚的分隔，所以你可以从头读到尾而完全不去运行它。

1. **安装依赖。** `aien` 包（本仓库）带着共享的 setup 辅助函数，并拉入 `groq` 客户端。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领免费 key（无需信用卡）；在 Colab 里通过**钥匙图标** → **Add new secret** 添加，名称严格写成 `GROQ_API_KEY`，并打开 notebook access。在本地则把 `GROQ_API_KEY` 设为环境变量。

（完整讲解见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup

# 加载 GROQ_API_KEY 并返回一个就绪的 Groq 客户端。只被下面的概念
# 部分使用；可选的 LoRA 附录跑在另一套 GPU 技术栈上。
client, MODEL = setup()

## 一条轴：你到底改动了模型的多少？

改变模型行为的四种方式排在同一条谱系上：**它们伸进模型内部的深度。** 这条轴值得记牢，因为它是*技术性*的，不随潮流移动：

```
 改的是 INPUTS（输入）                 改的是 WEIGHTS（权重）
 （模型不动）                                  （模型被修改）
 ┌────────────┬────────────┬─────────────────┬──────────────┐
 │ Prompting  │    RAG     │  LoRA / QLoRA   │ 全量微调 FFT │
 │ 指令与示例 │ 查询时检索 │   训练 <1% 的   │  重训每一个  │
 │            │  到的事实  │ 权重（adapter） │     权重     │
 └────────────┴────────────┴─────────────────┴──────────────┘
   ← 更便宜、更快、更易回退                     更强、更贵 →
```

**用词要精确。** 只有右半边才真正*改动模型*——它改变权重。Prompting 和 RAG 改的是模型的**输入**，权重始终不动。而 **LoRA 不是 fine-tuning 的同级选项，它是 fine-tuning 的*一种***（你仍然在训练权重，只是训练其中一小片低秩切片，而不是全部）。所以诚实的分类法是：改*输入*有两条路，而改*模型*的那件事有两个*深度*。

把这条谱系从左往右读，工作顺序就自然浮现了。**伸手去够能解决你问题的那根最左边的杠杆**，因为左边更便宜、迭代更快、也更容易回退：

1. **Prompting** 改的是*指令*。system prompt、few-shot、输出格式。零训练、即时迭代、不需要基础设施。
2. **RAG** 改的是*输入*。查询时检索相关事实塞进 prompt（整个第 03 节）。权重从不变动；你只是在喂更好的上下文。
3. **LoRA 微调** 改的是*一小片权重*。训练小型 adapter，把新的*行为*固化进去（后面详述）。
4. **全量微调** 改的是*全部*权重。对这个角色来说极少是正确的第一步；贵，而且很容易做得比一个好 prompt 还差。

> **🚩 常见错误 ——** 直接跳到右半边。它最贵、迭代最慢，而且因为听起来像 ML，人们会以为它才是「真正」的解决方案。AI 工程师的工作，通常是把 stakeholder 往这条谱系的*左边*劝，而不是右边。

这个 notebook 讲的就是精确地知道什么时候伸手进权重才是对的，下一个单元格会解释为什么这个门槛随时间*升高*了。

## 为什么「先伸手左边」是*现代*的答案 —— 钟摆

"Prompt before fine-tune" 不是永恒的定律；它是一个钟摆摆到了这个位置，而知道它*为什么*摆过来，才能让你在讨论中论证这次判断。

- **早期 LLM 时代（约 2020–2022）：fine-tuning 是默认做法。** 那时的基座模型指令遵循能力弱、上下文窗口小。想让模型在你的任务上稳定表现，就得微调；并没有一条足够强的纯 prompt 路径。托管微调 API 是当时的焦点。
- **如今：领域向*左*摆了。** 基座模型开箱就能很好地遵循指令，上下文窗口巨大（把示例/文档直接放进 prompt），RAG 也成熟为标准的知识注入方式。对最常见的需求——*「让模型知道或做我们的事」*——prompt + RAG 现在在成本、速度和新鲜度上都胜出。微调**在这类需求上变得不那么流行**，以至于 OpenAI 正在*把托管微调对新用户逐步关停*。
- **让你保持诚实的那个细微之处：微调没有消亡，它*专门化*了。** 它不再是知识类需求的首选，但对少数特定工作来说，它是对的工具，有些地方甚至还在*增长*：
  - **大规模下的风格 / 语气 / 格式**，那些在 prompt 里很难完全钉死的弥散行为。
  - **蒸馏一个小而便宜、快速的模型。** 微调一个小模型，让它在*某一个*窄任务上对齐被 prompt 的大模型，然后每次调用的成本和延迟只付很小一部分。这个用例正在*上升*，恰恰因为推理成本在规模化时很重要。

> **⭐ 关键收获 ——** 「fine-tune 之前先 prompt/RAG」今天成立，是因为*基座模型和围绕它们的工具链*变好了，而不是因为微调不好。面试时就按这个说法讲：先点出钟摆，再点出你仍会伸手往右边的具体条件（大规模风格、小模型蒸馏）。这就是背诵一条规则和理解它之间的区别。

## 决策，一张表

要内化的直觉：**prompting 和 RAG 改变模型*此刻知道什么*；fine-tuning 改变模型*默认怎么表现*。** 知识 → 上下文。行为 → 权重。

| 你想改变…… | 伸手去够 | 为什么 |
|---|---|---|
| 模型缺少的**事实 / 知识**（文档、私有数据、近期事件） | **RAG**（或 prompt） | 知识会变、且必须能引用；检索让它保持新鲜和落地事实。微调把事实按训练那一天固化进去：会过时、无法引用、更新昂贵。 |
| **格式 / 结构**（总是吐出这种 JSON 形状、这个 schema） | 先 **prompt**；若必须在规模化下万无一失再微调 | Prompting + tool calling（第 01 节）覆盖绝大部分；微调只在超大规模下消除最后几个百分点的格式漂移。 |
| **风格 / 语气 / 人设**（听起来像我们的品牌、我们的客服口吻） | **微调（LoRA）** | 风格是弥散行为，很难在 prompt 里完全写清；几百个示例比几段指令教得更好。 |
| **一个窄而重复的任务**要做得更便宜/更快（分类、抽取、海量改写） | **微调一个小模型** | 微调过的小模型可以在一个窄任务上对齐被 prompt 的大模型，每次调用成本和延迟只付一小部分。 |
| **基座模型压根做不到的新技能 / 推理** | **微调**，也许，但要狠狠验证 | 对这个角色很罕见，而且容易高估；通常换一个更好的基座模型或更好的 prompting 就先补上了。 |

从这张表里落出来两条规则：

- **微调是知识类需求的错误工具。** 这是头号误解。「模型不知道我们的产品文档 → 那我们拿它微调一下好了」几乎总是错的：你得到一个昂贵的模型，它*背下*了那些文档，却无法引用、不重训就更新不了。那是 RAG 的活。
- **它们能组合。** 真实系统常常在同一个模型上既为*风格/格式*做微调，又用 RAG 提供*知识*。不是二选一。

## LoRA / QLoRA 到底是什么

全量微调会更新模型里的*每一个*权重。对一个 7B 模型来说就是约 70 亿个数字，需要几十 GB 的 GPU 显存和一台硬核机器。这就是多年来微调门槛高不可攀的原因。

**LoRA（Low-Rank Adaptation）** 是改变这一切的技巧。它不更新那些大的权重矩阵，而是把它们*冻结*，在旁边训练很小的 "adapter" 矩阵，通常**不到参数量的 1%**。洞见在于：你需要的那份*变化*是低秩的（能用远少于完整矩阵的数字表达），所以一个小小的 adapter 就能承载它。你用一小部分显存和存储，拿到了全量微调的大部分质量。

- **LoRA** 冻结基座模型，训练小的低秩 adapter。adapter 只有几 MB；你可以为每个基座模型保留很多个并随时切换。
- **QLoRA** 在**量化**（4-bit）基座模型之上跑 LoRA。进一步削减显存，足以在一块免费档的 Colab GPU 上微调 7B 模型。
- **PEFT** 是 HuggingFace 的库（Parameter-Efficient Fine-Tuning），实现了 LoRA/QLoRA 等等。这是当今的标准工具链。

为什么这对你重要：LoRA 现在已经*主流*，不再是奇技淫巧。「我们会微调一个 LoRA」在这个岗位里是句平常话，而 adapter 足够便宜、已经是真实的选项，这让知道*它何时是错误的选项*更有价值，而不是更少。（有些推理服务商，包括 Groq，甚至能在推理时**服务** LoRA adapter，所以训练出来的 adapter 是可部署的，不只是实验室产物。）

## 第二条轴：全量微调 vs LoRA

顶部那条谱系在改权重的一侧放了*两个*格子，是有原因的。一旦你决定要微调，还有第二个决策：**怎么**更新权重，而它的实际后果比大多数人预想的更大。

把词分清楚：**fine-tuning 是统称**（在一个预训练模型上用你的数据做更新）；**全量微调（FFT）** 和 **LoRA** 是它下面的两种*方法*。FFT 更新每一个权重；LoRA 冻结基座，训练注入到各层里的、很小的低秩 adapter 矩阵。

| 方面 | 全量微调（FFT） | LoRA（Low-Rank Adaptation） |
|---|---|---|
| **如何工作** | 直接更新 100% 的参数 | 冻结基座权重；训练注入的小 adapter |
| **可训练参数** | 100%（例如 Llama-8B 约 8B） | 约 0.1–1%（数千万） |
| **显存 / GPU 成本** | 非常高：多 GPU，每个权重的 optimizer state + 梯度 | 低，单块（甚至消费级）GPU 就能装下 |
| **产出物** | 一整个新的多 GB 模型 checkpoint | 一个小 adapter 文件（往往 <100 MB） |
| **灾难性遗忘** | 风险更高；会侵蚀基座模型的通用知识 | 风险更低，因为基座权重保持冻结 |

**为什么 LoRA 在实践中是默认选择。** 在 LLM 工作中，LoRA（及其量化的表亲 QLoRA）才是团队真正会去够的方法，背后有两个超出「训练更便宜」的架构性理由：

- **多租户服务化。** 因为基座被冻结，你可以把一个大基座模型的*一份*副本载入 GPU 显存，然后**按请求热切换小小的 adapter**（一个用于法律摘要、一个用于代码、一个用于品牌语气），而不必为每个任务各起一个完整模型。AWS 的 [multi-LoRA-on-vLLM 文章](https://aws.amazon.com/blogs/machine-learning/efficiently-serve-dozens-of-fine-tuned-models-with-vllm-on-amazon-sagemaker-ai-and-amazon-bedrock/) 具体展示了这一点：adapter 在共享的冻结基座之上「按请求换进换出」，于是（他们的例子）五个各用约 10% GPU 的客户，被压到*单一*块共享 GPU 上，而不是五块闲置的 GPU。
- **可及性。** 现代 LLM 的 FFT 需要多 GPU 集群；LoRA 让一个团队用其中一小部分硬件就能适配一个开放权重模型。这很大程度上就是微调终于变得平易近人的原因（见下面那个在免费 T4 上跑 QLoRA 的附录）。

**什么时候选哪个：**

- **LoRA** 适合教一个本来就有能力的基座模型一种*风格、语气、结构化格式*（严格的 JSON/SQL），或某个领域任务。这是你实际会做的微调中的绝大多数。
- **全量微调** 适合剧烈、根本性的改变：新增一门语言、改变深层推理行为、大幅重塑领域表示，这些情况下薄薄的 adapter 不够用。对这个角色很罕见。

> **⭐ 关键收获 ——** 「我们要不要微调？」和「FFT 还是 LoRA？」是两个不同的问题。实践中第二个问题的答案几乎总是 **LoRA**，不只是为了省训练成本，更因为一个可以换 adapter 的冻结基座，在*服务化*上是根本性的更优方案。

> **⚠️ 生产现实 ——** adapter 放在哪里取决于平台，而这一点会坑人。多 LoRA 服务化（上面）让 adapter 保持**独立、按需加载**。但有些托管路径要求它们**先合并进基座**：例如 [Bedrock Custom Model Import](https://aws.amazon.com/blogs/machine-learning/amazon-bedrock-custom-model-import-now-generally-available/) 导入的是*完整合并后*的 Safetensors 权重，并且（原文）「期望把 adapter 合并进主 base model 权重」，而不是一个裸 adapter。同一个 LoRA，两种迥异的部署形态；要清楚你的目标平台期望哪一种。（下面服务商表里那条 Bedrock 备注，背后也是同一个事实。）

### 如果你*真的*要微调：托管 API vs 自己训练，以及谁能让你碰 LoRA

一旦评测说明微调挣得了它的位置，就有两种做法。而在设计讨论之前，有个微妙之处值得知道：**不是每个服务商都让你真正控制 LoRA。** 有些把训练方法整个抽象掉；另一些则把 adapter 变成你配置并部署的一等对象。下面所有片段**仅供参考，这里跑不了**（它们需要各家 SDK + 付费 key）；列出来是为了让你对*形态*熟悉。

**路径 1 —— 托管微调，方法被抽象掉（OpenAI、AWS Bedrock）。** 你上传一个 `.jsonl`，调一个方法，服务商负责训练*并*托管结果。你**不能**选 LoRA 还是全量微调；技术是隐藏在后面的。你选的是*目标*和几个训练旋钮，仅此而已。当你只是想要一个更好的模型、而不是一个研究旋钮时，这是常见的生产路径。

```python
# OpenAI —— 仅供参考。注意：OpenAI 正在对新用户逐步关停托管微调。
# 你选的是 OBJECTIVE（method=supervised|dpo|reinforcement）——从来不是 "LoRA"。
from openai import OpenAI
client = OpenAI()

f = client.files.create(file=open("train.jsonl", "rb"), purpose="fine-tune")
job = client.fine_tuning.jobs.create(
    model="gpt-4.1-mini-2025-04-14",
    training_file=f.id,
    method={"type": "supervised"},          # 你唯一能控制的「怎么做」；没有 rank/adapter
)
job = client.fine_tuning.jobs.retrieve(job.id)     # 轮询直到 status == "succeeded"
# 然后像任何模型一样调用它：
# client.chat.completions.create(model=job.fine_tuned_model, messages=[...])
```

```python
# AWS Bedrock —— 仅供参考。同样完全被抽象掉：没有 LoRA 类型，没有 lora_rank。
# customizationType 是 FINE_TUNING | CONTINUED_PRE_TRAINING | DISTILLATION | ...（没有 "LoRA"）
import boto3
bedrock = boto3.client("bedrock")           # 控制面

bedrock.create_model_customization_job(
    jobName="my-ft-job", customModelName="my-model",
    roleArn="arn:aws:iam::123456789012:role/BedrockCustomizationRole",
    baseModelIdentifier="meta.llama3-1-8b-instruct-v1:0:128k",
    customizationType="FINE_TUNING",
    hyperParameters={"epochCount": "5", "batchSize": "1", "learningRate": "0.0001"},
    trainingDataConfig={"s3Uri": "s3://my-bucket/train.jsonl"},
    outputDataConfig={"s3Uri": "s3://my-bucket/output/"},
)
# 之后这个自定义模型会被 DEPLOY（custom-model deployment 或 provisioned throughput），
# 并通过它的 ARN 调用。要把你自己的 LoRA 带进 Bedrock，你必须把它 MERGE 进基座权重、
# 导入完整模型——你不能上传一个裸 adapter。AWS 上显式、可控制 rank 的 LoRA 在 SageMaker，
# 不在 Bedrock。
```

**路径 2 —— 带*显式* LoRA 的托管微调（Together、Fireworks）。** 这些推理优先的平台把 LoRA adapter 变成一等公民：你选 **rank**（往往还有 alpha/dropout），它们训练 adapter，你把它部署到一个 endpoint 并调用。这才是真正「通过 API 造一个 LoRA 并使用」的路径。

```python
# Together AI —— 仅供参考。LoRA 是显式的，并且是默认的训练类型。
from together import Together
client = Together()

tf = client.files.upload(file="train.jsonl", purpose="fine-tune")
job = client.fine_tuning.create(
    training_file=tf.id,
    model="Qwen/Qwen3.5-9B",       # 查一下当前支持的模型列表
    lora=True,                     # <-- 你显式地要求 LoRA……
    lora_r=8, lora_alpha=16,       # ……并设置它的 rank/alpha（CLI + REST API 都会暴露）
    n_epochs=3,
)
# 完成后，把 adapter DEPLOY 到一个专用 endpoint，然后用 endpoint id 调用它：
#   tg beta endpoints deploy <MODEL_OBJECT_ID> --endpoint my-lora
#   Together(base_url="https://api-inference.together.ai/v1").chat.completions.create(
#       model="your-project-slug/my-lora", messages=[...])
```

```bash
# Fireworks AI —— 仅供参考（firectl CLI）。这里的监督微调就是 LoRA；
# 你直接设置 rank，并把 adapter 当作一等的模型资源来寻址。
firectl dataset create my-data ./train.jsonl
firectl sftj create --base-model <BASE_MODEL_ID> --dataset my-data \
                    --output-model my-lora --lora-rank 16      # <-- 显式的 LoRA rank
firectl deployment create "accounts/<ACCT>/models/my-lora"     # 部署这个 adapter
# 然后通过 OpenAI 兼容 API 调用 model="accounts/<ACCT>/models/my-lora"
```

**路径 3 —— 自己训练 adapter**，在你自己的 GPU 上用 `peft`/`trl`：完全控制（rank、目标模块、量化），跑在开放权重上，托管自定义模型没有按 token 的加价，但训练*和*服务化都由你负责。那就是本 notebook 末尾那个可选附录。

> **⭐ 关键收获 ——** 「我们会微调」这句话藏着一个真实的分叉。在 **OpenAI / Bedrock** 上方法被抽象掉了；你选不了 LoRA。在 **Together / Fireworks**（以及自托管的 `peft`）上，LoRA 是一个显式、可控制 rank 的 adapter，由你部署。在你向客户承诺「一个 LoRA」之前，先搞清楚你身处哪一种。

无论哪种，*决策*和*评测*都是同一件事，这正是这个 notebook 的全部要点。无论你是租训练还是自己跑，你只在 prompt + RAG 已经可证明地在评测里失败之后才微调，并且用第 02 节 / 第 04 节那套同样的 harness 来证明它确实有帮助。

## 成本，以及你实际会经历的那场争论

你迟早会遇到一个场合，有人问：*「我们干脆微调一下不行吗？」* 下面这张诚实的成本图，是拿来回答的——不是用来否决它，而是为了把顺序排对：

| | Prompt / RAG | LoRA 微调 |
|---|---|---|
| **前期成本** | 约等于零 | 一份标注数据集（几百到几千个示例）+ 一次 GPU 训练 |
| **迭代速度** | 秒级；改完重跑 | 小时级；重新整理数据、重训、重评测 |
| **更新知识** | 改检索到的文档 | 重训（知识被冻结进权重） |
| **规模化后的每次调用成本** | 每次调用都为上下文 token 付费 | 微调过的*小*模型每次调用可以便宜得多 |
| **能引用来源吗？** | 能（RAG） | 不能 |
| **主要风险** | prompt 膨胀、检索质量 | 过拟合、知识过时、一套你根本没建的评测 |

专业的做法是 **prompt → RAG → fine-tune，而且只有更便宜的那根杠杆在评测里明确失败时，才往前走一步。** 没有评测（第 04 节）就去做微调，是团队花掉一周 GPU 时间、把模型弄得*感觉*更好、却*可测量地*更差的方式。如果你从这个 notebook 只带走一件事去面试：你能解释*为什么通常该先伸手去 RAG，以及在哪些具体条件下你仍会做微调*（规模化下的风格/格式/延迟，极少是知识）。

In [ ]:
# 一个你确实能在 Groq 上跑的概念检验：基座模型在一个好 prompt 下，是否已经
# 把这个任务做得足够好？诚实地回答它，是任何微调之前的门槛。
# 这里是个格式/风格任务，正是人们会想去微调的那一类。
examples = [
    "ticket: wifi keeps dropping in the west conference room since tuesday",
    "ticket: cant log into the vpn, mfa code never arrives",
]
SYSTEM = ('You are a support triage bot. For each ticket reply with ONE line: '
          '"<PRIORITY> | <CATEGORY> | <one-line summary>". '
          'PRIORITY in {P1,P2,P3}. CATEGORY in {network,auth,hardware,other}. Nothing else.')

for t in examples:
    r = client.chat.completions.create(
        model=MODEL, max_tokens=60,
        messages=[{'role': 'system', 'content': SYSTEM},
                  {'role': 'user', 'content': t}],
    )
    print(r.choices[0].message.content.strip())

# 如果这样一个 prompt 已经能在你真实的工单分布上稳稳拿下格式，
# 那你不需要为格式去做微调。只有当 prompt 失败得足够频繁、量足够大，
# 把它固化进去才划算时，微调才挣得它的位置。
print('\n-> Prompt got the format. Now ask: does it hold on 500 real tickets? '
      'That eval — not a hunch — is what justifies (or kills) a fine-tune.')

---

## 可选附录：一次真实的 LoRA 微调（需要 GPU runtime）

上面所有内容都跑在 Groq 上。**这一节不是。** 训练权重需要一块 GPU 和另一套技术栈（`peft`、`transformers`、`trl`、`bitsandbytes`），它们都不碰 Groq。把它当成一个你*可以*跑的实验室，用来把 LoRA 变得具体，而不是 Groq 路径的一部分。

**怎么跑：** 在 Colab 里 **Runtime → Change runtime type → T4 GPU**，然后运行下面的单元格。在 CPU runtime 上，防护单元格会用一条清楚的提示拦住你，而不是崩掉。它在一个小开放模型上训练一个极小的 LoRA adapter，所以在免费 T4 上几分钟就能跑完；这是*机制*演示，不是质量结果。

如果你只是来拿那个判断的（也就是这个 notebook 的要点），读到上面那行就可以停了。你已经拿到了。

In [ ]:
# GUARD：这个附录需要 GPU。在 CPU 上就早一点、清楚地停下，而不是
# 装一大堆重依赖，然后在训练循环深处失败。
import torch
assert torch.cuda.is_available(), (
    'No GPU detected. This appendix requires one: Runtime -> Change runtime type '
    '-> T4 GPU, then re-run. (The concept sections above do not need this.)'
)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 沉重的、仅 GPU 的依赖。不属于 Groq 技术栈；这里是故意安装的。
# 当前 peft 要求 torchao>=0.16；Colab 自带的是旧版，所以在这里把它固定上去。
%pip install -q -U transformers peft trl datasets bitsandbytes accelerate "torchao>=0.16.0"

In [ ]:
# 一次极小的 LoRA 微调，从头到尾，跑在一个小开放模型上。
# 任务：用一小撮示例教会一种固定的回复 STYLE（简短、带前缀）——
# 风格恰恰是微调擅长、而 prompting 难以钉住的东西。
#
# 这个单元格是照着能在免费 T4 上跑、并且真的展示出风格迁移来写的。
# 注释标出了真正要紧的选择（每一个都是让它跑起来时的真实失败模式）：
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

BASE = 'HuggingFaceTB/SmolLM2-135M-Instruct'  # 很小：在 T4 上几分钟就能训练完

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
# fp32，不是 fp16：一个 135M 模型用 fp16 训练往往产出垃圾/NaN，以及退化的
# 贪心输出。fp32 在这里能轻松装进 T4，而且在这个量级上稳定得多。
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32, device_map='auto')

# 玩具数据集：教一种自家风格 -> 每个回答都以 "ACME> " 开头，并保持简短。
raw = [
    ("How do I reset my password?", "ACME> Settings > Security > Reset password. Done."),
    ("Is the API down?", "ACME> Check status.acme.io. If red, we're on it."),
    ("How do I export my data?", "ACME> Settings > Data > Export. You'll get a CSV by email."),
    ("Can I get a refund?", "ACME> Yes within 30 days. Reply here with your order id."),
] * 16  # 重复，让这个小模型真的能抓到这个模式

IM_END = tok.convert_tokens_to_ids("<|im_end|>")

def build(q, a):
    # Prompt 用与推理时相同的 template + add_generation_prompt 构建，
    # 这样训练与推理的格式逐字节一致（这里不一致 = adapter 永远不触发）。
    enc = tok.apply_chat_template([{'role': 'user', 'content': q}],
                                  add_generation_prompt=True, tokenize=True, return_dict=True)
    prompt_ids = list(enc['input_ids'])
    # 答案以 <|im_end|> 结尾、不带末尾换行，好让模型学会 STOP——否则
    # 它永远不吐停止 token，然后一圈圈刷出无穷无尽的换行 token。
    answer_ids = tok(a, add_special_tokens=False)['input_ids'] + [IM_END]
    input_ids = prompt_ids + answer_ids
    # 只在答案上算 loss（prompt 的 labels = -100）。没有这一点，模型会把容量花在
    # 复读 system 前导（"...SmolLM..."）上，而不是学 ACME 风格。
    labels = [-100] * len(prompt_ids) + answer_ids
    return {'input_ids': input_ids, 'labels': labels, 'attention_mask': [1] * len(input_ids)}

ds = Dataset.from_list([build(q, a) for q, a in raw])
print('one training example (decoded):\n', repr(tok.decode(ds[0]['input_ids'])))

# 给 LoRA 多一点触达范围（所有 attention 投影），让 135M 模型吃下这个风格。
peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                      target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
                      task_type='CAUSAL_LM')
model = get_peft_model(model, peft_cfg)

def collate(feats):
    maxlen = max(len(f['input_ids']) for f in feats)
    pad = lambda xs, v: [x + [v] * (maxlen - len(x)) for x in xs]
    return {'input_ids': torch.tensor(pad([f['input_ids'] for f in feats], tok.pad_token_id)),
            'labels': torch.tensor(pad([f['labels'] for f in feats], -100)),
            'attention_mask': torch.tensor(pad([f['attention_mask'] for f in feats], 0))}

trainer = Trainer(
    model=model,
    args=TrainingArguments(output_dir='lora-out', num_train_epochs=10,
                           per_device_train_batch_size=4, learning_rate=3e-4,
                           logging_steps=10, report_to='none'),
    train_dataset=ds, data_collator=collate)
trainer.train()
model = trainer.model
print('\nTrained a LoRA adapter with',
      sum(p.numel() for p in model.parameters() if p.requires_grad),
      'trainable params (vs', sum(p.numel() for p in model.parameters()), 'total).')

In [ ]:
# adapter 之前/之后，同一个 prompt（一个模型训练时从没见过的）。adapter
# 应该把回复推向基座模型本来不用的「ACME> ..., 简短」自家风格。
def reply(prompt):
    enc = tok.apply_chat_template([{'role': 'user', 'content': prompt}],
                                  add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True).to(model.device)
    n_prompt = enc['input_ids'].shape[1]
    # repetition_penalty + min_new_tokens 防止小模型在开口之前就塌成一串
    # 换行；eos = <|im_end|>，好让它在回合结束时干净地停下。
    out = model.generate(**enc, max_new_tokens=40, do_sample=False,
                         repetition_penalty=1.3, min_new_tokens=5,
                         eos_token_id=tok.convert_tokens_to_ids("<|im_end|>"),
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][n_prompt:], skip_special_tokens=True).strip()

q = 'How do I change my email address?'
with model.disable_adapter():          # adapter 关闭 -> 基座模型的默认口吻
    print('BASE :', reply(q))
print('LoRA :', reply(q))              # adapter 打开 -> 学到的 "ACME> ..." 自家风格

print('\nThis is a mechanics demo on a 135M model and a toy dataset -- do not read quality '
      'into it. The point: <1% of params, a few minutes, a STYLE the base model did not have, '
      'learned from a handful of examples and toggled on/off with one adapter. Whether it BEATS '
      'a good prompt is an eval question (section 04), every time.')

## 练习

1. **在面试里把它论证出来。** 每题用 3–4 句话回答：(a) *「客户想让模型知道他们的内部 wiki：微调还是 RAG？」* (b) *「他们想让每个回答都用他们的品牌语气：微调还是 prompt？」* (c) *「他们每天分类 200 万张工单，GPT-4 级延迟太慢：你会怎么建议？」* 这些都是真实的 FDE 筛选问题。
2. **用 prompt 干掉一次微调。** 拿附录里那个 house-style 任务，试着只用 prompting + few-shot，让*基座*模型（关掉 adapter，或任何 Groq 模型）对上「ACME> terse」风格。你能做到多接近？这是任何微调之前，诚实的第一步。
3. **给这个决策算成本。** 对一个每天 100 万次调用的窄分类任务，估算每月成本：(a) prompt 一个大模型 vs (b) 一个微调过的小模型（复用 `01-model-apis/04-context-and-caching` 里的成本算法）。在多大的量级上，微调的前期成本才能回本？
4. **FFT 还是 LoRA？** 对每一种情况，用一句话说出你会选哪种方法以及为什么：(a) 对上某家律所的 house drafting 风格；(b) 教模型一门它几乎不会的低资源语言；(c) 输出严格的内部 schema JSON，漂移接近零；(d) 改变模型的核心推理行为。哪些属于那种「罕见的」全量微调用例？
5. **设计服务化方案。** 一个客户需要同一个基座模型的*八个*部门专用变体（客服、法务、销售……），每个流量都不高。对比八个全量微调模型与一个冻结基座加八个热切换 LoRA adapter 的成本（见 multi-LoRA 那篇文章）。为什么冻结基座的设计在 GPU 利用率上胜出，而如果其中一个变体需要*剧烈的*行为改变，隐患又是什么？
6. **（GPU）改变 LoRA 的 rank。** 用 `r=1` 和 `r=64` 重跑附录。更高的 rank 改变的是学到的风格，还是只是可训练参数量？把它跟低秩那个洞见联系起来：多小的 `r` 仍然能抓住这个任务？